### Building Agentic Document Understanding

In [ ]:
from PIL import Image
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any

#### 1. Text Extraction with PaddleOCR + LayoutLM Ordering

In [ ]:
from paddleocr import PaddleOCR
# Initialize once per kernel to avoid PDX reinit errors
if "ocr" not in globals():
    ocr = PaddleOCR(lang="en", enable_mkldnn=False,  # prevents MKLDNN/PIR crash
                    )
    
# Load image
image_path = "files/page_0.png"
display(Image.open(image_path))

In [ ]:
# Run OCR 
result = ocr.predict(image_path)
page = result[0]

In [ ]:
page.keys()

In [ ]:
texts = page['rec_texts']      # recognized text strings
scores = page['rec_scores']    # confidence scores
boxes = page['rec_polys']      # bounding box coordinates

In [ ]:
for text, score, box in list(zip(texts, scores, boxes))[:20]:
    coords = box.astype(int).tolist()
    print(f"{text:50} | {score:.3f} | {coords}")

##### 1.2 OCR bounding boxes

In [ ]:
processed_img = page['doc_preprocessor_res']['output_img']  # preprocessed image used for OCR
img_plot = processed_img.copy()
show_text = False

In [ ]:


for text, box in zip(texts, boxes):
    pts = np.array(box, dtype=int)
    cv2.polylines(img_plot, [pts], True, (0, 255, 0), 2)
    x, y = pts[0]
    if show_text:
        cv2.putText(img_plot, text, (x, y - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Aligned Bounding Boxes (Processed Image)")
plt.show()

##### 1.3. Structuring OCR Results with a Dataclass

In [ ]:
@dataclass
class OCRRegion:
    text: str
    bbox: list
    confidence: float # [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]

    @property 
    def bbox_xyxy(self):
        """return bbox in [x_min, y_min, x_max, y_max] format"""
        x_coords = [point[0] for point in self.bbox]
        y_coords = [point[1] for point in self.bbox]
        return [min(x_coords), min(y_coords), max(x_coords), max(y_coords)]

In [ ]:
ocr_regions: List[OCRRegion] = []
for text, score, box in zip(texts, scores, boxes):
    ocr_regions.append(OCRRegion(text=text, bbox=box.astype(int).tolist(), confidence=score))

len(ocr_regions) # number of text regions detected

##### 1.4. LayoutLM Reading Order

In [ ]:
import os
os.getcwd()

In [ ]:
import pdfplumber

with pdfplumber.open("data/statements/TD_CASH_BACK_VISA_INFINITE__CARD_3226_Feb_06-2026.pdf") as pdf:
    for page in pdf.pages:
        print(page.extract_text())

In [ ]:
page = pdf.pages[0]
page = page.filter(lambda o: o.get("upright", True))

hdr = page.search("MONTANT")[0]
col = page.crop((0, hdr["bottom"], hdr["x1"] + 15, page.height))
text = col.extract_text(x_tolerance=1)



In [ ]:
import re

ROW = re.compile(
    r"^(?P<date>\d{1,2} [a-zéû]{3,4})\s+(?P<posted>\d{1,2} [a-zéû]{3,4})\s+"
    r"(?P<desc>.+?)\s+(?P<amount>-?\d{1,3}(?: \d{3})*,\d{2})(?: ?\$)?$"
)


In [ ]:
import re
from decimal import Decimal

ROW = re.compile(
    r"^(?P<date>\d{1,2} [a-zéû]{3,4})\s+"       # "5 jan", "30 jan", "2 fév"
    r"(?P<posted>\d{1,2} [a-zéû]{3,4})\s+"      # posting date, same shape
    r"(?P<desc>.+?)\s+"                         # description (lazy: as short as possible)
    r"(?P<amount>-?\d{1,3}(?: \d{3})*,\d{2})"   # "5,98", "-1 200,00"
    r"(?: ?\$)?$"                               # optional "$", then end of line
)
CONTINUATION = re.compile(r"[A-Z][A-Z \-]{2,25}")  # "MONTREAL", "TORONTO ON"

def parse_rows(text: str) -> list[dict]:
    transactions = []
    for line in text.splitlines():
        line = line.strip()
        m = ROW.match(line)
        if m:
            transactions.append(m.groupdict())
        elif transactions and CONTINUATION.fullmatch(line):
            transactions[-1]["desc"] += " " + line   # wrapped description
        # anything else (headers, "Suite", totals) is skipped
    return transactions


In [ ]:
MONTHS = {"jan": 1, "fév": 2, "mar": 3, "avr": 4, "mai": 5, "juin": 6,
          "juil": 7, "aoû": 8, "sep": 9, "oct": 10, "nov": 11, "déc": 12}

def to_amount(s: str) -> Decimal:
    return Decimal(s.replace(" ", "").replace(",", "."))

rows = parse_rows(text)
total = sum(to_amount(r["amount"]) for r in rows)
print(total == Decimal("882.38") - Decimal("1478.53"))  # True → nothing missed


In [ ]:
for line in text.splitlines():
    if re.match(r"^\d{1,2} [a-zéû]{3,4}", line.strip()) and not ROW.match(line.strip()):
        print("starts like a row but didn't match:", repr(line))


##### 1.5. Expenses table → pandas DataFrame (pdfplumber + regex, all pages)

In [ ]:
import re
import pdfplumber
import pandas as pd

pdf_path = "data/statements/TD_CASH_BACK_VISA_INFINITE__CARD_3226_Feb_06-2026.pdf"

MONTHS = {"jan": 1, "fév": 2, "mar": 3, "avr": 4, "mai": 5, "juin": 6,
          "juil": 7, "aoû": 8, "sep": 9, "oct": 10, "nov": 11, "déc": 12}
DATE = r"\d{1,2} [a-zéû]{3,4}"
AMOUNT = r"-?\d{1,3}(?: \d{3})*,\d{2}"
ROW = re.compile(rf"^(?P<date>{DATE})\s+(?P<posted>{DATE})\s+(?P<desc>.+?)\s+(?P<amount>{AMOUNT})(?: ?\$)?$")
CONTINUATION = re.compile(r"[A-Z][A-Z \-]{2,25}")       # wrapped description, e.g. "MONTREAL"
TABLE_END = re.compile(r"^(Sui|TOTAL|NOUVEAU SOLDE)")  # "Suite" (cropped) / totals end the table

def to_amount(s: str) -> float:
    return float(s.replace("\xa0", "").replace(" ", "").replace(",", "."))

rows, full_text = [], ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page = page.filter(lambda o: o.get("upright", True))  # drop rotated margin text
        full_text += (page.extract_text(x_tolerance=1) or "") + "\n"
        hdr = page.search("MONTANT")
        if not hdr:
            continue  # no transaction table on this page
        # keep only the transaction column, below the header (drops the side panel)
        col = page.crop((0, hdr[0]["bottom"], hdr[0]["x1"] + 15, page.height))
        after_row = False  # a wrapped description can only directly follow a transaction
        for line in (col.extract_text(x_tolerance=1) or "").splitlines():
            line = line.replace("\xa0", " ").strip()
            if TABLE_END.match(line):
                break
            if m := ROW.match(line):
                rows.append(m.groupdict() | {"page": page.page_number})
                after_row = True
            elif after_row and CONTINUATION.fullmatch(line):
                rows[-1]["desc"] += " " + line
                after_row = False
            else:
                after_row = False

# dates have no year: take it from "PÉRIODE DE RELEVÉ : du 07 jan 2026 au 06 fév 2026"
period = re.search(rf"du ({DATE}) (\d{{4}}) au ({DATE}) (\d{{4}})", full_text)
end_month, end_year = MONTHS[period.group(3).split()[1]], int(period.group(4))

def to_date(s: str) -> pd.Timestamp:
    day, mon = s.split()
    month = MONTHS[mon]
    year = end_year - 1 if month > end_month else end_year  # e.g. "28 déc" on a January statement
    return pd.Timestamp(year, month, int(day))

df = pd.DataFrame(rows)
df["date"] = df["date"].map(to_date)
df["posted"] = df["posted"].map(to_date)
df["amount"] = df["amount"].map(to_amount)
df["kind"] = "purchase"
df.loc[df["desc"].str.contains("PAIEMENT"), "kind"] = "payment"
df.loc[df["desc"].str.contains("PRIME"), "kind"] = "reward"
df.loc[df["desc"].str.contains("FRAIS"), "kind"] = "fee"
df = df.rename(columns={"desc": "description"})[["date", "posted", "description", "amount", "kind", "page"]]

# reconcile: previous balance + transactions must equal the new balance
opening = to_amount(re.search(rf"SOLDE DU RELEVÉ PRÉCÉDENT ({AMOUNT})", full_text).group(1))
closing = to_amount(re.search(rf"NOUVEAU SOLDE ({AMOUNT})", full_text).group(1))
print(f"{len(df)} transactions | sum {df['amount'].sum():.2f} | expected {closing - opening:.2f} | "
      f"reconciled: {round(opening + df['amount'].sum(), 2) == closing}")

expenses = df[df["kind"].isin(["purchase", "fee"])]
print(f"total spending: {expenses['amount'].sum():.2f} $")
df